# OpenWild-BoarNet — revision experiments (IJIES Paper ID 20266330)

One notebook that produces every number, table and figure requested by
Reviewer 1, Reviewer 2 and the Editor. Run it ONCE with "Save Version ->
Save & Run All (Commit)". Every stage checkpoints to /kaggle/working, so if
the session dies you can re-run and finished stages are skipped.

Before the GPU run, do a free smoke test: Accelerator = None, SMOKE_TEST = True.

Kaggle settings: Internet ON, Accelerator 'GPU T4 x2' (NOT P100), inputs:
  1. angadahir/wild-boar-dataset-yolo-annotated            (known class)
  2. ankanghosh651/object-detection-wildlife-dataset-yolo-format (unknown)
  3. iamsouravbanerjee/animal-image-dataset-90-different-animals
     (cross-source boar + near-OOD species; optional but strongly advised)

In [ ]:
# %% ===================== 1. SETTINGS (edit only this cell) =====================
SMOKE_TEST = False          # True = tiny CPU dry-run (no GPU quota) to catch errors
YOLO_SEEDS = [0, 1, 2]      # R1-5: repeated training seeds
YOLO_EPOCHS = 50
YOLO_PATIENCE = 10
YOLO_IMGSZ = 640
YOLO_BATCH = 16
BOOTSTRAP_B = 2000          # R1-5 / R2-6: bootstrap resamples
CALIB_REPEATS = 20          # R1-5: prototype/Weibull/threshold variability
RUN_STAGES = {              # set False to skip a stage (outputs are reused)
    "check": True,
    "discover": True,
    "yolo_train": True,
    "yolo_eval": True,
    "features": True,
    "analysis": True,
    "package": True,
}

In [ ]:
# %% ===================== 2. INSTALL (keeps Kaggle torch/numpy untouched) =====
import os, sys, subprocess, importlib, importlib.util, importlib.metadata

IN_KAGGLE = os.path.exists("/kaggle/working")
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"


def _pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("pip FAILED:", " ".join(args), "\n", r.stderr[-1500:])
    return r.returncode == 0


def _ver(pkg):
    try:
        return importlib.metadata.version(pkg)
    except Exception:
        return None


def _healthy():
    """Import the scientific stack in a SEPARATE process so a broken numpy cannot crash this kernel."""
    r = subprocess.run([sys.executable, "-c",
                        "import numpy, numpy.random, pandas, scipy.stats, sklearn; print('numpy', numpy.__version__)"],
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-800:]


if IN_KAGGLE and not os.environ.get("OWB_SKIP_INSTALL"):
    _ok, _msg = _healthy()
    if not _ok:
        # A previous pip install left numpy half-replaced ("numpy.dtype size changed").
        print("Damaged NumPy detected -> repairing.\n", _msg)
        for _ in range(3):
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "numpy"], capture_output=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall",
                        "numpy>=2.0,<2.3"], capture_output=True)
        _ok, _msg = _healthy()
        print("Repair", "succeeded:" if _ok else "FAILED:", _msg)
        if _ok:
            print("\n>>> Kernel will now restart. When it is back, press 'Run All' again. <<<")
            sys.stdout.flush()
            os._exit(0)
        raise SystemExit("Environment still damaged. Use 'Run > Factory reset' (or stop the session and "
                         "start a new one), then Run All.")
    print("Scientific stack OK:", _msg)

if IN_KAGGLE and not os.environ.get("OWB_SKIP_INSTALL"):
    # NEVER replace torch / numpy shipped with the Kaggle image: mixing binary builds breaks
    # numpy ("numpy.dtype size changed"). Freeze them with a pip constraints file and only
    # add the packages that are missing.
    _keep = {p: _ver(p) for p in ["numpy", "torch", "torchvision", "scipy", "pandas",
                                  "scikit-learn", "pillow", "matplotlib", "opencv-python"]}
    _cons = "/kaggle/working/pip_constraints.txt"
    with open(_cons, "w") as _f:
        _f.write("\n".join(f"{p}=={v}" for p, v in _keep.items() if v))
    print("Frozen:", {k: v for k, v in _keep.items() if v})
    if importlib.util.find_spec("ultralytics") is None:
        _pip("ultralytics>=8.3", "-c", _cons)
    if importlib.util.find_spec("timm") is None:
        _pip("timm", "-c", _cons)
    if importlib.util.find_spec("open_clip") is None:
        _pip("open_clip_torch", "-c", _cons)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "ray", "wandb"],
                   capture_output=True)
    _changed = {p: (v, _ver(p)) for p, v in _keep.items() if v and _ver(p) != v}
    if _changed:
        raise SystemExit(f"pip changed core packages {_changed}. Use 'Run > Factory reset', then Run All.")
    _ok, _msg = _healthy()
    if not _ok:
        raise SystemExit("Scientific stack broken after installs. Use 'Run > Factory reset', then Run All.\n" + _msg)

In [ ]:
# %% ===================== 3. CONFIG, PATHS, LOGGING ============================
import json, time, random, math, glob, shutil, traceback, warnings, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

WORK = Path("/kaggle/working") if IN_KAGGLE else Path(os.environ.get("OWB_WORK", "./owb_work")).resolve()
INPUT_ROOT = Path(os.environ.get("OWB_INPUT", "/kaggle/input"))
OUT = WORK / ("revision_outputs_smoke" if SMOKE_TEST else "revision_outputs")
D_TAB, D_FIG, D_FEAT, D_YOLO, D_LOG = (OUT / "tables", OUT / "figures", OUT / "features",
                                        OUT / "yolo", OUT / "logs")
for _d in (OUT, D_TAB, D_FIG, D_FEAT, D_YOLO, D_LOG):
    _d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

if SMOKE_TEST:
    YOLO_SEEDS, YOLO_EPOCHS, YOLO_IMGSZ, YOLO_BATCH = [0], 1, 320, 8
    BOOTSTRAP_B, CALIB_REPEATS = 100, 3

# --- method hyper-parameter grids (selected on the VALIDATION set only) ---
TAILS = [0.10, 0.15, 0.20, 0.25, 0.30]      # Weibull extreme-tail fractions
PAPER_TAIL = 0.20                           # value used in the submitted paper
KNN_K = [1, 5, 10, 50]
MP_K = [2, 4, 8, 16]                        # multi-prototype EVT
OCSVM_NU = [0.05, 0.10]
TPR_TARGET = 0.95                           # P1: accept 95% of known validation images
YOLO_CONF = 0.25
ROI_PAD = 0.10
DUP_SIM = 0.95                              # near-duplicate cosine similarity
SMOKE_N = 24
NUM_WORKERS = min(4, os.cpu_count() or 2)
FEAT_BATCH = 64

NEAR_OOD = ["pig", "hippopotamus", "bison", "ox", "cow", "bear", "badger", "hyena",
            "goat", "sheep", "deer", "wombat", "porcupine", "dog"]
FAR_EXT = ["elephant", "zebra", "rhinoceros"]
KNOWN_EXT = ["boar"]

BACKBONES = ["vit_b16", "dinov2_b14", "bioclip"]
VIEWS = ["full", "gtroi", "bg", "yoloroi"]
REF = ("vit_b16", "full", "evt_proto")      # the method of the submitted paper

LOG_FILE = D_LOG / "run_log.txt"


def log(*a):
    msg = time.strftime("%H:%M:%S ") + " ".join(str(x) for x in a)
    print(msg, flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(msg + "\n")


STATUS_FILE = D_LOG / "stage_status.json"


def run_stage(name, fn):
    status = json.loads(STATUS_FILE.read_text()) if STATUS_FILE.exists() else {}
    if not RUN_STAGES.get(name, True):
        log(f"[{name}] skipped by RUN_STAGES")
        return
    t0 = time.time()
    log(f"===== [{name}] start =====")
    try:
        fn()
        status[name] = {"ok": True, "minutes": round((time.time() - t0) / 60, 2)}
        log(f"===== [{name}] done in {status[name]['minutes']} min =====")
    except Exception as e:
        status[name] = {"ok": False, "error": repr(e), "minutes": round((time.time() - t0) / 60, 2)}
        log(f"!!!!! [{name}] FAILED: {e!r}\n{traceback.format_exc()}")
    STATUS_FILE.write_text(json.dumps(status, indent=2))


def cuda_ok():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False


def stage_check():
    """Fail fast (in minutes, not hours) if the environment is broken."""
    import torch
    log("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
        "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
    a = torch.from_numpy(np.arange(6, dtype=np.float32)).view(2, 3)   # numpy<->torch bridge
    if torch.cuda.is_available():
        cap = torch.cuda.get_device_capability(0)
        arch = f"sm_{cap[0]}{cap[1]}"
        if arch not in torch.cuda.get_arch_list():
            raise RuntimeError(f"This torch build ({torch.__version__}) does not support "
                               f"{torch.cuda.get_device_name(0)} ({arch}). In Kaggle choose "
                               f"Accelerator = 'GPU T4 x2' instead of P100.")
        assert float((a.cuda() @ a.cuda().T).sum()) == float((a @ a.T).sum())
    import ultralytics, timm, sklearn, scipy
    log("ultralytics", ultralytics.__version__, "| timm", timm.__version__, "| numpy", np.__version__,
        "| sklearn", sklearn.__version__, "| scipy", scipy.__version__)
    try:
        import open_clip
        log("open_clip", open_clip.__version__)
    except Exception as e:
        log("open_clip not importable (BioCLIP baseline will be skipped):", repr(e))
    if not SMOKE_TEST and not torch.cuda.is_available():
        log("WARNING: full run without GPU will be extremely slow")

In [ ]:
# %% ===================== 4. DATA DISCOVERY -> manifest.csv =====================
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_images(d):
    d = Path(d)
    if not d.is_dir():
        return []
    return sorted(str(p) for p in d.iterdir() if p.suffix.lower() in IMG_EXT)


def label_path_for(img):
    parts = list(Path(img).parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            break
    return str(Path(*parts).with_suffix(".txt"))


def read_yolo_labels(lp):
    """Return list of [cls, xc, yc, w, h] (normalised). Polygons -> bounding box."""
    out = []
    if not os.path.exists(lp):
        return out
    with open(lp) as f:
        for line in f:
            v = line.strip().split()
            if len(v) < 5:
                continue
            c = int(float(v[0]))
            nums = [float(x) for x in v[1:]]
            if len(nums) == 4:
                xc, yc, w, h = nums
            else:  # segmentation polygon
                xs, ys = nums[0::2], nums[1::2]
                x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
                xc, yc, w, h = (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
            if w > 0 and h > 0:
                out.append([c, xc, yc, w, h])
    return out


def box_cover(boxes, g=100):
    if not boxes:
        return 0.0
    m = np.zeros((g, g), bool)
    for _, xc, yc, w, h in boxes:
        x1, x2 = int(max(0, (xc - w / 2)) * g), int(math.ceil(min(1, (xc + w / 2)) * g))
        y1, y2 = int(max(0, (yc - h / 2)) * g), int(math.ceil(min(1, (yc + h / 2)) * g))
        m[y1:y2, x1:x2] = True
    return float(m.mean())


def _walk_dirs(root, max_depth=8):
    root = Path(root)
    base = len(root.parts)
    for dp, dn, _ in os.walk(root):
        p = Path(dp)
        if len(p.parts) - base > max_depth:
            dn[:] = []
            continue
        # do not descend into leaf image folders (speed)
        dn[:] = [d for d in dn if d not in ("images", "labels")] if (p / "images").is_dir() else dn
        yield p


def discover_roots():
    boar, wild, ani = [], [], []
    for p in _walk_dirs(INPUT_ROOT):
        if (p / "images" / "train").is_dir() and (p / "labels" / "train").is_dir() and (p / "images" / "test").is_dir():
            boar.append(p)
        if (p / "train" / "images").is_dir() and (p / "test" / "images").is_dir() and \
                ((p / "valid" / "images").is_dir() or (p / "val" / "images").is_dir()):
            wild.append(p)
        if p.name.lower() == "boar" and p.parent.is_dir():
            n_sub = sum(1 for q in p.parent.iterdir() if q.is_dir())
            if n_sub >= 50:
                ani.append(p.parent)
    pick = lambda c, key: (sorted(c, key=lambda x: (key not in str(x).lower(), len(str(x))))[0] if c else None)
    b = pick(boar, "boar")
    w = pick([x for x in wild if x != b], "wild")
    a = pick(ani, "animal")
    return b, w, a


def wildlife_names(wroot):
    import yaml
    for y in list(Path(wroot).glob("*.y*ml")) + list(Path(wroot).parent.glob("*.y*ml")):
        try:
            d = yaml.safe_load(open(y))
            n = d.get("names")
            if isinstance(n, dict):
                return {int(k): str(v).lower() for k, v in n.items()}
            if isinstance(n, list):
                return {i: str(v).lower() for i, v in enumerate(n)}
        except Exception:
            pass
    log("WARNING: wildlife yaml not found; using fallback class names")
    return {0: "buffalo", 1: "elephant", 2: "rhino", 3: "zebra"}


def stage_discover():
    broot, wroot, aroot = discover_roots()
    log("Known (wild boar) root :", broot)
    log("Unknown wildlife root  :", wroot)
    log("90-animals root        :", aroot)
    assert broot is not None, "Wild boar dataset not found under /kaggle/input"
    assert wroot is not None, "Wildlife dataset not found under /kaggle/input"
    names = wildlife_names(wroot)
    log("Wildlife class names:", names)
    rows = []

    def add(path, source, split, group, species, boxes):
        rows.append(dict(path=path, source=source, split=split, group=group, species=species,
                         boxes=json.dumps(boxes), n_boxes=len(boxes), has_box=int(len(boxes) > 0),
                         box_cover=round(box_cover(boxes), 4)))

    for split in ["train", "val", "test"]:
        imgs = list_images(broot / "images" / split)
        if SMOKE_TEST:
            imgs = imgs[:SMOKE_N]
        for im in imgs:
            add(im, "boar_main", split, "known", "wild_boar", read_yolo_labels(label_path_for(im)))
    for split_in, split in [("valid", "val"), ("val", "val"), ("test", "test")]:
        d = wroot / split_in / "images"
        if not d.is_dir():
            continue
        imgs = list_images(d)
        if SMOKE_TEST:
            imgs = imgs[:SMOKE_N]
        for im in imgs:
            bx = read_yolo_labels(label_path_for(im))
            cls = sorted({b[0] for b in bx})
            sp = names.get(cls[0], f"class{cls[0]}") if len(cls) == 1 else ("mixed" if cls else "none")
            add(im, "wild_main", split, "unknown", sp, bx)
    if aroot is not None:
        for cname, grp in [(c, "known_ext") for c in KNOWN_EXT] + [(c, "near_ood") for c in NEAR_OOD] + \
                          [(c, "far_ext") for c in FAR_EXT]:
            d = next((q for q in aroot.iterdir() if q.is_dir() and q.name.lower() == cname), None)
            if d is None:
                log(f"  90-animals class '{cname}' not found - skipped")
                continue
            imgs = list_images(d)
            if SMOKE_TEST:
                imgs = imgs[:4]
            for im in imgs:
                add(im, "ani90", "ext", grp, cname, [])
    else:
        log("WARNING: 90-animals dataset not attached -> near-OOD / cross-source tests will be skipped")
    man = pd.DataFrame(rows)
    man.insert(0, "img_id", np.arange(len(man)))
    man.to_csv(OUT / "manifest.csv", index=False)
    summ = man.groupby(["source", "split", "group", "species"]).size().reset_index(name="n_images")
    summ.to_csv(D_TAB / "T00_dataset_manifest_summary.csv", index=False)
    log("\n" + summ.to_string())
    json.dump({"boar_root": str(broot), "wild_root": str(wroot),
               "ani90_root": str(aroot) if aroot else None, "wildlife_names": names},
              open(OUT / "data_roots.json", "w"), indent=2)


def load_manifest():
    return pd.read_csv(OUT / "manifest.csv")

In [ ]:
# %% ===================== 5. YOLOv8n: TRAIN (3 seeds, resumable) =================
def yolo_data_yaml():
    import yaml
    roots = json.load(open(OUT / "data_roots.json"))
    broot = Path(roots["boar_root"])
    if SMOKE_TEST:  # build a tiny symlinked copy so CPU validation is fast
        man = load_manifest()
        mini = WORK / "smoke_yolo_data"
        for split in ["train", "val", "test"]:
            for sub in ["images", "labels"]:
                (mini / sub / split).mkdir(parents=True, exist_ok=True)
            for im in man[(man.source == "boar_main") & (man.split == split)].path:
                dst = mini / "images" / split / Path(im).name
                if not dst.exists():
                    os.symlink(im, dst)
                lp = label_path_for(im)
                dl = mini / "labels" / split / Path(lp).name
                if os.path.exists(lp) and not dl.exists():
                    shutil.copy(lp, dl)
        broot = mini
    y = {"path": str(broot), "train": "images/train", "val": "images/val",
         "test": "images/test", "names": {0: "wild_boar"}}
    p = OUT / "wild_boar_data.yaml"
    with open(p, "w") as f:
        yaml.dump(y, f)
    return str(p)


def _yolo_settings():
    try:
        from ultralytics import settings
        settings.update({k: False for k in ["wandb", "mlflow", "comet", "clearml", "dvc",
                                             "neptune", "raytune", "hub"] if k in settings})
    except Exception as e:
        log("ultralytics settings update skipped:", e)


def stage_yolo_train():
    from ultralytics import YOLO
    _yolo_settings()
    data = yolo_data_yaml()
    dev = 0 if cuda_ok() else "cpu"
    for s in YOLO_SEEDS:
        run = D_YOLO / f"seed{s}"
        done = run / "DONE"
        last, best = run / "weights" / "last.pt", run / "weights" / "best.pt"
        if done.exists():
            log(f"YOLO seed {s}: already trained")
            continue
        t0 = time.time()
        try:
            if last.exists():
                log(f"YOLO seed {s}: resuming from {last}")
                YOLO(str(last)).train(resume=True)
            else:
                log(f"YOLO seed {s}: training from yolov8n.pt")
                YOLO("yolov8n.pt").train(
                    data=data, epochs=YOLO_EPOCHS, patience=YOLO_PATIENCE, imgsz=YOLO_IMGSZ,
                    batch=YOLO_BATCH, workers=NUM_WORKERS, seed=s, deterministic=True,
                    pretrained=True, optimizer="auto", project=str(D_YOLO), name=f"seed{s}",
                    exist_ok=True, device=dev, cache=(not SMOKE_TEST), plots=True,
                    fraction=1.0, verbose=True)
        except AssertionError as e:  # "nothing to resume" -> training already finished
            log(f"YOLO seed {s}: resume assertion ({e}); treating as finished")
        if best.exists():
            done.write_text(json.dumps({"minutes": round((time.time() - t0) / 60, 1)}))
            log(f"YOLO seed {s}: finished")
        else:
            raise RuntimeError(f"best.pt missing for seed {s}")

In [ ]:
# %% ===================== 6. YOLOv8n: EVALUATE + PREDICT ON ALL IMAGES ============
def stage_yolo_eval():
    from ultralytics import YOLO
    _yolo_settings()
    data = yolo_data_yaml()
    man = load_manifest()
    dev = 0 if cuda_ok() else "cpu"
    rows = []
    for s in YOLO_SEEDS:
        best = D_YOLO / f"seed{s}" / "weights" / "best.pt"
        if not best.exists():
            log(f"seed {s}: no best.pt, skipped")
            continue
        m = YOLO(str(best))
        # (a) detection metrics. conf=0.001 is the standard mAP protocol;
        #     conf=0.25 reproduces the protocol of the submitted paper.
        for split in ["val", "test"]:
            for conf in [0.001, 0.25]:
                r = m.val(data=data, split=split, imgsz=YOLO_IMGSZ, batch=YOLO_BATCH, conf=conf,
                          iou=0.7, device=dev, plots=(split == "test" and conf == 0.001),
                          project=str(D_YOLO / "eval"), name=f"seed{s}_{split}_conf{conf}",
                          exist_ok=True, verbose=False)
                rows.append(dict(seed=s, split=split, conf=conf, precision=r.box.mp, recall=r.box.mr,
                                 mAP50=r.box.map50, mAP50_95=r.box.map,
                                 ms_per_img_inference=r.speed.get("inference", np.nan)))
        # (b) predictions on every image (train only for seed 0: needed for YOLO crops)
        sub = man if s == YOLO_SEEDS[0] else man[man.split != "train"]
        pf = D_YOLO / f"pred_seed{s}.csv"
        if pf.exists():
            log(f"seed {s}: predictions exist")
            continue
        recs = []
        paths = sub.path.tolist()
        ids = sub.img_id.tolist()
        for i0 in range(0, len(paths), 256):
            chunk = paths[i0:i0 + 256]
            try:
                res = list(m.predict(source=chunk, conf=0.01, iou=0.7, imgsz=YOLO_IMGSZ, device=dev,
                                     half=cuda_ok(), verbose=False, stream=True, max_det=50))
            except Exception as e:  # fall back to one-by-one so one bad file cannot kill the run
                log("predict chunk failed, falling back:", e)
                res = []
                for pth in chunk:
                    try:
                        res.append(m.predict(source=pth, conf=0.01, iou=0.7, imgsz=YOLO_IMGSZ,
                                             device=dev, verbose=False, max_det=50)[0])
                    except Exception:
                        res.append(None)
            for iid, r in zip(ids[i0:i0 + 256], res):
                if r is None or r.boxes is None or len(r.boxes) == 0:
                    recs.append(dict(img_id=iid, max_conf=0.0, n_det_025=0, x1=np.nan, y1=np.nan,
                                     x2=np.nan, y2=np.nan, n_det_total=0))
                    continue
                cf = r.boxes.conf.cpu().numpy()
                xy = r.boxes.xyxyn.cpu().numpy()
                j = int(cf.argmax())
                recs.append(dict(img_id=iid, max_conf=float(cf[j]), n_det_025=int((cf >= YOLO_CONF).sum()),
                                 x1=float(xy[j, 0]), y1=float(xy[j, 1]), x2=float(xy[j, 2]),
                                 y2=float(xy[j, 3]), n_det_total=len(cf)))
            log(f"seed {s}: predicted {min(i0 + 256, len(paths))}/{len(paths)}")
        pd.DataFrame(recs).to_csv(pf, index=False)
    if rows:
        df = pd.DataFrame(rows)
        old = D_TAB / "T01_yolo_detection_per_seed.csv"
        df.to_csv(old, index=False)

In [ ]:
# %% ===================== 7. FEATURE EXTRACTION (3 backbones x 4 views) ==========
def _crop(img, box_xyxyn, pad=ROI_PAD):
    W, H = img.size
    x1, y1, x2, y2 = box_xyxyn
    bw, bh = (x2 - x1), (y2 - y1)
    x1, x2 = max(0.0, x1 - pad * bw), min(1.0, x2 + pad * bw)
    y1, y2 = max(0.0, y1 - pad * bh), min(1.0, y2 + pad * bh)
    X1, Y1, X2, Y2 = int(x1 * W), int(y1 * H), int(math.ceil(x2 * W)), int(math.ceil(y2 * H))
    if X2 - X1 < 8 or Y2 - Y1 < 8:
        return img
    return img.crop((X1, Y1, X2, Y2))


def stage_features():
    import torch
    from PIL import Image, ImageOps, ImageDraw
    man = load_manifest()
    N = len(man)
    need = [(b, v) for b in BACKBONES for v in VIEWS if not (D_FEAT / f"{b}__{v}.npy").exists()]
    if not need:
        log("features: all present")
        return
    ypf = D_YOLO / f"pred_seed{YOLO_SEEDS[0]}.csv"
    if ypf.exists():
        yp = pd.read_csv(ypf).set_index("img_id")
    else:
        log("WARNING: YOLO predictions missing -> 'yoloroi' view will equal full image")
        yp = pd.DataFrame(columns=["n_det_025", "x1", "y1", "x2", "y2"])
    yboxes = {}
    for iid in man.img_id:
        if iid in yp.index and yp.loc[iid, "n_det_025"] > 0:
            r = yp.loc[iid]
            yboxes[iid] = (r.x1, r.y1, r.x2, r.y2)
    MEAN_RGB = (124, 116, 104)

    class DS(torch.utils.data.Dataset):
        def __len__(self):
            return N

        def __getitem__(self, i):
            row = man.iloc[i]
            ok = 1
            try:
                img = ImageOps.exif_transpose(Image.open(row.path)).convert("RGB")
            except Exception:
                img, ok = Image.new("RGB", (224, 224)), 0
            W, H = img.size
            boxes = json.loads(row.boxes)
            xyxy = [(xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2) for _, xc, yc, w, h in boxes]
            if xyxy:
                big = max(xyxy, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
                gt = _crop(img, big)
                bg = img.copy()
                dr = ImageDraw.Draw(bg)
                for b in xyxy:
                    bw, bh = b[2] - b[0], b[3] - b[1]
                    dr.rectangle([max(0, (b[0] - .05 * bw)) * W, max(0, (b[1] - .05 * bh)) * H,
                                  min(1, (b[2] + .05 * bw)) * W, min(1, (b[3] + .05 * bh)) * H], fill=MEAN_RGB)
            else:
                gt, bg = img, img
            yb = yboxes.get(int(row.img_id))
            yr = _crop(img, yb) if yb is not None else img
            ts = [torch.from_numpy(np.asarray(x.resize((224, 224), Image.BICUBIC), dtype=np.uint8).copy()
                                   ).permute(2, 0, 1) for x in (img, gt, bg, yr)]
            return torch.stack(ts), i, ok

    dev = "cuda" if cuda_ok() else "cpu"
    models = {}
    import timm

    def add_timm(key, name, **kw):
        try:
            m = timm.create_model(name, pretrained=True, num_classes=0, **kw).eval().to(dev)
            cfg = timm.data.resolve_model_data_config(m)
            models[key] = (m, torch.tensor(cfg["mean"]).view(1, 3, 1, 1).to(dev),
                           torch.tensor(cfg["std"]).view(1, 3, 1, 1).to(dev), lambda mm, x: mm(x))
            log(f"loaded {key}: {name} mean={cfg['mean']} std={cfg['std']}")
        except Exception as e:
            log(f"!! could not load {key} ({name}): {e!r}")

    add_timm("vit_b16", "vit_base_patch16_224")              # same backbone as the paper
    add_timm("dinov2_b14", "vit_base_patch14_dinov2.lvd142m", img_size=224)
    try:
        import open_clip
        m, _, pre = open_clip.create_model_and_transforms("hf-hub:imageomics/bioclip")
        nm = next(t for t in pre.transforms if t.__class__.__name__ == "Normalize")
        models["bioclip"] = (m.eval().to(dev), torch.tensor(nm.mean).view(1, 3, 1, 1).to(dev),
                             torch.tensor(nm.std).view(1, 3, 1, 1).to(dev), lambda mm, x: mm.encode_image(x))
        log("loaded bioclip: hf-hub:imageomics/bioclip")
    except Exception as e:
        log(f"!! could not load BioCLIP: {e!r}")
    assert "vit_b16" in models, "ViT-B/16 failed to load - check Internet is ON"

    feats = {}
    ok_flags = np.ones(N, int)
    dl = torch.utils.data.DataLoader(DS(), batch_size=FEAT_BATCH, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=(dev == "cuda"))
    done = 0
    t0 = time.time()
    with torch.no_grad():
        for x, idx, ok in dl:
            B = x.shape[0]
            ok_flags[idx.numpy()] = ok.numpy()
            x = x.to(dev, non_blocking=True).float().div_(255.0).view(B * 4, 3, 224, 224)
            for key, (m, mu, sd, fwd) in models.items():
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(dev == "cuda")):
                    f = fwd(m, (x - mu) / sd)
                f = f.float().view(B, 4, -1).cpu().numpy()
                if key not in feats:
                    feats[key] = np.zeros((N, 4, f.shape[-1]), np.float32)
                feats[key][idx.numpy()] = f
            done += B
            if done % (FEAT_BATCH * 20) == 0 or done == N:
                log(f"features {done}/{N}  ({(time.time() - t0) / 60:.1f} min)")
    for key, F in feats.items():
        for vi, v in enumerate(VIEWS):
            np.save(D_FEAT / f"{key}__{v}.npy", F[:, vi, :])
    np.save(D_FEAT / "load_ok.npy", ok_flags)
    log("unreadable images:", int((ok_flags == 0).sum()))

In [ ]:
# %% ===================== 8. STATISTICS & SCORING UTILITIES =====================
from scipy.stats import weibull_min, kstest, rankdata, binomtest


def l2n(X):
    return X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)


def counts_to_metrics(TP, FN, TN, FP):
    """Positive class = UNKNOWN intrusion. Works on scalars or arrays."""
    TP, FN, TN, FP = [np.asarray(a, float) for a in (TP, FN, TN, FP)]
    with np.errstate(divide="ignore", invalid="ignore"):
        nk, nu = TN + FP, TP + FN
        N = nk + nu
        acc = (TP + TN) / N
        kar, urr = TN / nk, TP / nu
        f1u = 2 * TP / (2 * TP + FP + FN)
        f1k = 2 * TN / (2 * TN + FN + FP)
        den = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
        mcc = np.where(den > 0, (TP * TN - FP * FN) / np.where(den > 0, den, 1), 0.0)
        pe = ((TP + FP) * (TP + FN) + (TN + FN) * (TN + FP)) / N ** 2
        kappa = (acc - pe) / (1 - pe)
    return {"accuracy": acc, "balanced_accuracy": (kar + urr) / 2, "macro_f1": (f1u + f1k) / 2,
            "f1_known": f1k, "f1_unknown": f1u, "mcc": mcc, "kappa": kappa,
            "known_recall": kar, "unknown_recall": urr, "unknown_far": 1 - urr, "known_frr": 1 - kar}


def auroc_rows(sk, su):
    a = np.concatenate([sk, su], axis=1)
    r = rankdata(a, axis=1)
    nk, nu = sk.shape[1], su.shape[1]
    return (r[:, nk:].sum(1) - nu * (nu + 1) / 2) / (nk * nu)


def auroc(sk, su):
    return float(auroc_rows(np.asarray(sk)[None], np.asarray(su)[None])[0])


def aupr(sk, su):
    from sklearn.metrics import average_precision_score
    y = np.r_[np.zeros(len(sk)), np.ones(len(su))]
    return float(average_precision_score(y, np.r_[sk, su]))


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


class Boot:
    """Stratified bootstrap indices, shared by all methods -> paired comparisons."""

    def __init__(self, nk, nu, B, seed=SEED):
        rng = np.random.default_rng(seed)
        self.ik = rng.integers(0, nk, (B, nk))
        self.iu = rng.integers(0, nu, (B, nu))


def evaluate(sk, su, tau, boot=None):
    sk, su = np.asarray(sk, float), np.asarray(su, float)
    rk, ru = sk > tau, su > tau
    TP, FP = ru.sum(), rk.sum()
    m = {k: float(v) for k, v in counts_to_metrics(TP, len(su) - TP, len(sk) - FP, FP).items()}
    m.update(TP=int(TP), FN=int(len(su) - TP), TN=int(len(sk) - FP), FP=int(FP),
             n_known=len(sk), n_unknown=len(su), tau=float(tau),
             auroc=auroc(sk, su), aupr=aupr(sk, su),
             unknown_far_at_95known_oracle=float(np.mean(su <= np.quantile(sk, 0.95))))
    arrays = {}
    if boot is not None:
        RK, RU = rk[boot.ik], ru[boot.iu]
        tp, fp = RU.sum(1), RK.sum(1)
        bm = counts_to_metrics(tp, su.size - tp, sk.size - fp, fp)
        bm["auroc"] = auroc_rows(sk[boot.ik], su[boot.iu])
        for k, v in bm.items():
            m[k + "_lo"], m[k + "_hi"] = [float(x) for x in np.nanpercentile(v, [2.5, 97.5])]
        arrays = {k: bm[k] for k in ["auroc", "macro_f1", "balanced_accuracy", "mcc",
                                      "unknown_recall", "known_recall"]}
        arrays["correct"] = None
    return m, arrays


def tau_p1(s_cal_known):
    return float(np.quantile(s_cal_known, TPR_TARGET))


def tau_youden(sk, su):
    cand = np.unique(np.r_[sk, su])
    sks, sus = np.sort(sk), np.sort(su)
    kar = np.searchsorted(sks, cand, side="right") / len(sk)
    urr = 1 - np.searchsorted(sus, cand, side="right") / len(su)
    return float(cand[int(np.argmax((kar + urr) / 2))])


# --- Weibull EVT on cosine distance to the class prototype -----------------
def fit_weibull_tail(d_train, frac):
    m = max(5, int(round(frac * len(d_train))))
    tail = np.sort(d_train)[-m:]
    c, loc, scale = weibull_min.fit(tail, floc=0)
    ks = kstest(tail, "weibull_min", args=(c, loc, scale)).statistic
    return {"shape_k": float(c), "loc": float(loc), "scale_lambda": float(scale),
            "tail_n": int(m), "tail_frac": frac, "ks": float(ks)}


def wcdf(d, p):
    return weibull_min.cdf(d, p["shape_k"], p["loc"], p["scale_lambda"])


def proto_dist(X, tr):
    mu = X[tr].mean(0)
    return 1 - l2n(X) @ (mu / np.linalg.norm(mu))


def knn_scores(Z, tr, ks):
    bank = Z[tr]
    out = {k: np.zeros(len(Z)) for k in ks}
    kmax = max(ks)
    for i in range(0, len(Z), 2048):
        S = Z[i:i + 2048] @ bank.T
        kk = min(kmax, bank.shape[0])
        top = -np.sort(-np.partition(S, -kk, axis=1)[:, -kk:], axis=1)
        for k in ks:
            out[k][i:i + 2048] = 1 - top[:, min(k, kk) - 1]
    return out


def mp_evt_scores(Z, tr, K, tail, seed=SEED, n_init=10):
    from sklearn.cluster import KMeans
    K = int(min(K, max(1, len(tr) // 10)))
    km = KMeans(K, n_init=n_init, random_state=seed).fit(Z[tr])
    C = l2n(km.cluster_centers_)
    D = 1 - Z @ C.T
    lab = km.labels_
    dtr = D[tr][np.arange(len(tr)), lab]
    gp = fit_weibull_tail(dtr, tail)
    cdfs = []
    min_members = 10 if SMOKE_TEST else 50
    for c in range(K):
        dc = dtr[lab == c]
        p = fit_weibull_tail(dc, tail) if len(dc) >= min_members else gp
        cdfs.append(wcdf(D[:, c], p))
    return np.min(np.stack(cdfs, 1), 1), K


def family_scores(X, tr, heavy):
    """All candidate scorers. Higher score = more likely UNKNOWN.
    Returns {family: [(config, score, extra), ...]}."""
    from sklearn.covariance import LedoitWolf
    from sklearn.decomposition import PCA
    from sklearn.svm import OneClassSVM
    from sklearn.ensemble import IsolationForest
    Z = l2n(X)
    out = {}
    d = proto_dist(X, tr)
    dtr = d[tr]
    out["cos_proto"] = [("-", d, {"pct95_train": float(np.quantile(dtr, 0.95))})]
    out["zscore_proto"] = [("-", (d - dtr.mean()) / dtr.std(), {})]
    evt = []
    for t in TAILS:
        p = fit_weibull_tail(dtr, t)
        evt.append((f"tail={t:.2f}", wcdf(d, p), p))
    out["evt_proto_grid"] = evt
    best_ks = min(evt, key=lambda e: e[2]["ks"])          # unknown-free tail rule (R1-5)
    out["evt_proto"] = [best_ks]
    out["evt_proto_paper"] = [e for e in evt if abs(e[2]["tail_frac"] - PAPER_TAIL) < 1e-9]
    lw = LedoitWolf().fit(Z[tr])
    out["mahalanobis"] = [("ledoit-wolf", lw.mahalanobis(Z), {})]
    kn = knn_scores(Z, tr, KNN_K)
    out["knn"] = [(f"k={k}", kn[k], {"k": k}) for k in KNN_K]
    tail_sel = best_ks[2]["tail_frac"]
    mp = []
    for K in MP_K:
        s, Ku = mp_evt_scores(Z, tr, K, tail_sel, n_init=(2 if SMOKE_TEST else 10))
        mp.append((f"K={Ku},tail={tail_sel:.2f}", s, {"K": Ku}))
    out["mp_evt"] = mp
    if heavy:
        pca = PCA(n_components=int(min(128, len(tr) - 1, Z.shape[1])), random_state=SEED).fit(Z[tr])
        P = pca.transform(Z)
        oc = []
        for nu in OCSVM_NU:
            o = OneClassSVM(kernel="rbf", gamma="scale", nu=nu).fit(P[tr])
            oc.append((f"nu={nu}", -o.decision_function(P), {"nu": nu}))
        out["ocsvm"] = oc
        iso = IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1).fit(P[tr])
        out["iforest"] = [("n=300", -iso.score_samples(P), {})]
    return out

In [ ]:
# %% ===================== 9. ANALYSIS (CPU only) =================================
from contextlib import contextmanager


@contextmanager
def guard(name):
    """A failing analysis block is logged and skipped; later blocks still run."""
    t0 = time.time()
    try:
        yield
        log(f"   block '{name}' ok ({time.time() - t0:.1f}s)")
    except Exception as e:
        log(f"!! analysis block '{name}' FAILED: {e!r}\n{traceback.format_exc()}")


def stage_analysis():
    man = load_manifest()
    N = len(man)
    g = lambda m: np.where(m.values)[0]
    I = {
        "train": g((man.source == "boar_main") & (man.split == "train")),
        "cal_k": g((man.source == "boar_main") & (man.split == "val")),
        "cal_u": g((man.source == "wild_main") & (man.split == "val")),
        "te_k": g((man.source == "boar_main") & (man.split == "test")),
        "te_u": g((man.source == "wild_main") & (man.split == "test")),
        "known_ext": g(man.group == "known_ext"),
        "near_ood": g(man.group == "near_ood"),
        "far_ext": g(man.group == "far_ext"),
    }
    log({k: len(v) for k, v in I.items()})
    assert len(I["train"]) and len(I["cal_k"]) and len(I["cal_u"]) and len(I["te_k"]) and len(I["te_u"])
    has_ext = len(I["known_ext"]) + len(I["near_ood"]) + len(I["far_ext"]) > 0
    boot = Boot(len(I["te_k"]), len(I["te_u"]), BOOTSTRAP_B)

    # ---------------- YOLO-based scores ----------------
    yolo = {}
    for s in YOLO_SEEDS:
        pf = D_YOLO / f"pred_seed{s}.csv"
        if pf.exists():
            p = pd.read_csv(pf).set_index("img_id").reindex(man.img_id)
            yolo[s] = p
    scores = {}       # (backbone, view, family) -> dict(score, config, extra)
    grid_rows = []

    def sel_by_val(cands):
        best = None
        for cfg, s, ex in cands:
            a = auroc(s[I["cal_k"]], s[I["cal_u"]])
            if best is None or a > best[0]:
                best = (a, cfg, s, ex)
        return best[1], best[2], best[3], best[0]

    if yolo:
        y0 = yolo[YOLO_SEEDS[0]]
        s_y = 1 - y0.max_conf.fillna(0).values
        scores[("yolov8n", "image", "yolo_conf")] = dict(score=s_y, config="1-max_conf", extra={},
                                                         fixed_tau=1 - YOLO_CONF)
    feat_files = sorted(D_FEAT.glob("*__*.npy"))
    avail = {}
    for f in feat_files:
        b, v = f.stem.split("__")
        avail[(b, v)] = np.load(f)
    log("feature sets:", list(avail))
    ext_invalid = (man.source == "ani90").values
    for (b, v), X in avail.items():
        t0 = time.time()
        heavy = (b == "vit_b16" and v in ("full", "gtroi"))
        fam = family_scores(X.astype(np.float64), I["train"], heavy)
        for fname, cands in fam.items():
            if fname == "evt_proto_grid":
                continue
            if fname in ("evt_proto", "evt_proto_paper", "cos_proto", "zscore_proto", "mahalanobis", "iforest"):
                cfg, s, ex = cands[0]
                va = auroc(s[I["cal_k"]], s[I["cal_u"]])
            else:
                cfg, s, ex, va = sel_by_val(cands)
            s = s.astype(float).copy()
            if v in ("gtroi", "bg"):
                s[ext_invalid] = np.nan       # no ground-truth boxes for external images
            fixed = None
            if fname in ("evt_proto", "evt_proto_paper"):
                fixed = 0.5
            elif fname == "zscore_proto":
                fixed = 1.645
            elif fname == "cos_proto":
                fixed = ex["pct95_train"]
            scores[(b, v, fname)] = dict(score=s, config=cfg, extra=ex, fixed_tau=fixed, val_auroc=va)
            # grid rows (ablation over hyper-parameters, all on TEST at P1 & P2)
            grid_list = fam["evt_proto_grid"] if fname == "evt_proto" else (cands if len(cands) > 1 else [])
            for c2, s2, ex2 in grid_list:
                for proto in ["P1_val_known_TPR95", "P2_val_youden"]:
                    tau = tau_p1(s2[I["cal_k"]]) if proto.startswith("P1") else tau_youden(s2[I["cal_k"]], s2[I["cal_u"]])
                    mt, _ = evaluate(s2[I["te_k"]], s2[I["te_u"]], tau, boot)
                    grid_rows.append(dict(backbone=b, view=v, family=("evt_proto" if fname == "evt_proto" else fname),
                                          config=c2, protocol=proto,
                                          val_auroc=auroc(s2[I["cal_k"]], s2[I["cal_u"]]),
                                          weibull_ks=ex2.get("ks", np.nan),
                                          weibull_k=ex2.get("shape_k", np.nan),
                                          weibull_lambda=ex2.get("scale_lambda", np.nan), **mt))
                if fname == "evt_proto":
                    mt, _ = evaluate(s2[I["te_k"]], s2[I["te_u"]], 0.5, boot)
                    grid_rows.append(dict(backbone=b, view=v, family="evt_proto", config=c2,
                                          protocol="P0_fixed_tau0.5", val_auroc=np.nan,
                                          weibull_ks=ex2["ks"], weibull_k=ex2["shape_k"],
                                          weibull_lambda=ex2["scale_lambda"], **mt))
        log(f"scored {b}/{v} in {time.time() - t0:.1f}s")

    # integrated detector-gated pipeline: YOLO box -> ViT crop -> EVT; no box -> reject
    if yolo:
        y0 = yolo[YOLO_SEEDS[0]]
        nodet = (y0.n_det_025.fillna(0).values == 0)
        for b in BACKBONES:
            k = (b, "yoloroi", "evt_proto")
            if k in scores:
                s = scores[k]["score"].copy()
                s[nodet] = 2.0
                scores[(b, "yoloroi", "pipeline_yolo_evt")] = dict(score=s, config=scores[k]["config"] + "+gate",
                                                                   extra={}, fixed_tau=0.5)
    pd.DataFrame(grid_rows).to_csv(D_TAB / "T10_hyperparameter_grid_TEST.csv", index=False)

    # ---------------- main results: every (backbone, view, family) x protocol ----------------
    rows, boot_arrays, taus = [], {}, {}
    for key, d in scores.items():
        s = d["score"]
        protos = {"P1_val_known_TPR95": tau_p1(s[I["cal_k"]]),
                  "P2_val_youden": tau_youden(s[I["cal_k"]], s[I["cal_u"]])}
        if d.get("fixed_tau") is not None:
            protos["P0_fixed_no_tuning"] = d["fixed_tau"]
        for pn, tau in protos.items():
            mt, arr = evaluate(s[I["te_k"]], s[I["te_u"]], tau, boot)
            mc, _ = evaluate(s[I["cal_k"]], s[I["cal_u"]], tau, None)
            rows.append(dict(backbone=key[0], view=key[1], family=key[2], config=d["config"], protocol=pn,
                             val_auroc=mc["auroc"], val_balanced_accuracy=mc["balanced_accuracy"],
                             val_known_recall=mc["known_recall"], val_unknown_recall=mc["unknown_recall"],
                             **{("test_" + k if not k.startswith("test_") else k): v for k, v in mt.items()}))
            taus[(key, pn)] = tau
            if pn == "P1_val_known_TPR95":
                arr["correct"] = np.r_[~(s[I["te_k"]] > tau), (s[I["te_u"]] > tau)]
                boot_arrays[key] = arr
    res = pd.DataFrame(rows)
    res.to_csv(D_TAB / "T02_all_methods_TEST_with_CI.csv", index=False)

    def fmt(r, k, pct=True):
        f = 100 if pct else 1
        dg = 2 if pct else 4
        return f"{r['test_' + k] * f:.{dg}f} [{r['test_' + k + '_lo'] * f:.{dg}f}, {r['test_' + k + '_hi'] * f:.{dg}f}]"

    compact = []
    for _, r in res.iterrows():
        compact.append(dict(backbone=r.backbone, view=r.view, method=r.family, config=r.config,
                            protocol=r.protocol, tau=round(r.test_tau, 4),
                            AUROC=fmt(r, "auroc", False), Accuracy=fmt(r, "accuracy"),
                            Balanced_Acc=fmt(r, "balanced_accuracy"), Macro_F1=fmt(r, "macro_f1"),
                            MCC=fmt(r, "mcc", False), Kappa=fmt(r, "kappa", False),
                            Known_Recall=fmt(r, "known_recall"), Unknown_Recall=fmt(r, "unknown_recall"),
                            Unknown_FAR=fmt(r, "unknown_far"), Known_FRR=fmt(r, "known_frr"),
                            AUPR=round(r.test_aupr, 4)))
    pd.DataFrame(compact).to_csv(D_TAB / "T02b_all_methods_TEST_formatted.csv", index=False)

    sp_te = man.species.values
    curves = []
    # ---------------- paired comparison vs. the submitted method ----------------
    with guard('paired comparison vs. the submitted method'):
        pair_rows = []
        if REF in boot_arrays:
            ra = boot_arrays[REF]
            for key, a in boot_arrays.items():
                if key == REF:
                    continue
                row = dict(backbone=key[0], view=key[1], family=key[2], reference="/".join(REF))
                for met in ["auroc", "macro_f1", "balanced_accuracy", "mcc"]:
                    dlt = a[met] - ra[met]
                    dlt = dlt[~np.isnan(dlt)]
                    row[f"delta_{met}"] = float(np.mean(dlt)) if len(dlt) else np.nan
                    row[f"delta_{met}_lo"], row[f"delta_{met}_hi"] = (np.percentile(dlt, [2.5, 97.5]) if len(dlt) else (np.nan, np.nan))
                    row[f"p_{met}"] = float(min(1.0, 2 * min(np.mean(dlt <= 0), np.mean(dlt >= 0)))) if len(dlt) else np.nan
                c_ref, c_oth = ra["correct"], a["correct"]
                bb, cc = int((c_ref & ~c_oth).sum()), int((~c_ref & c_oth).sum())
                row.update(mcnemar_ref_only_correct=bb, mcnemar_other_only_correct=cc,
                           mcnemar_p=float(binomtest(bb, bb + cc, 0.5).pvalue) if bb + cc > 0 else 1.0)
                pair_rows.append(row)
        pd.DataFrame(pair_rows).to_csv(D_TAB / "T03_paired_tests_vs_submitted_method.csv", index=False)

    # ---------------- per-species analysis (R1-3) ----------------
    with guard('per-species analysis (R1-3)'):
        sp_rows = []
        sp_te = man.species.values
        key_methods = [k for k in scores if k[2] in ("evt_proto", "knn", "mp_evt", "mahalanobis", "yolo_conf",
                                                     "pipeline_yolo_evt") and k[1] in ("full", "gtroi", "yoloroi", "image")]
        for key in key_methods:
            s = scores[key]["score"]
            for pn in ["P1_val_known_TPR95", "P2_val_youden", "P0_fixed_no_tuning"]:
                if (key, pn) not in taus:
                    continue
                tau = taus[(key, pn)]
                for split, idx in [("test", I["te_u"]), ("val", I["cal_u"])]:
                    for sp in sorted(set(sp_te[idx])):
                        ii = idx[sp_te[idx] == sp]
                        k_rej = int((s[ii] > tau).sum())
                        lo, hi = wilson(k_rej, len(ii))
                        sp_rows.append(dict(backbone=key[0], view=key[1], family=key[2], protocol=pn,
                                            split=split, species=sp, n=len(ii), rejected_as_unknown=k_rej,
                                            unknown_recall=k_rej / len(ii), ci_lo=lo, ci_hi=hi,
                                            auroc_vs_known_test=auroc(s[I["te_k"]], s[ii])))
        pd.DataFrame(sp_rows).to_csv(D_TAB / "T04_species_wise_unknown_recall.csv", index=False)

        # YOLO closed-set exposure per seed (image level, conf 0.25)
        yrows = []
        for sd, p in yolo.items():
            det = (p.n_det_025.fillna(0).values > 0)
            for grp, idx in [("known_test", I["te_k"]), ("unknown_test", I["te_u"]), ("unknown_val", I["cal_u"]),
                             ("known_ext", I["known_ext"]), ("near_ood", I["near_ood"]), ("far_ext", I["far_ext"])]:
                if len(idx) == 0:
                    continue
                n_fa = int(det[idx].sum())
                confs = p.max_conf.values[idx][det[idx]]
                yrows.append(dict(seed=sd, group=grp, species="ALL", n=len(idx), images_with_boar_detection=n_fa,
                                  rate=n_fa / len(idx), ci_lo=wilson(n_fa, len(idx))[0], ci_hi=wilson(n_fa, len(idx))[1],
                                  n_false_boxes_ge025=int(p.n_det_025.fillna(0).values[idx].sum()),
                                  mean_max_conf_detected=float(confs.mean()) if len(confs) else np.nan,
                                  max_conf=float(confs.max()) if len(confs) else np.nan))
                for sp in sorted(set(sp_te[idx])):
                    ii = idx[sp_te[idx] == sp]
                    k = int(det[ii].sum())
                    yrows.append(dict(seed=sd, group=grp, species=sp, n=len(ii), images_with_boar_detection=k,
                                      rate=k / len(ii), ci_lo=wilson(k, len(ii))[0], ci_hi=wilson(k, len(ii))[1]))
        ydf = pd.DataFrame(yrows)
        ydf.to_csv(D_TAB / "T05_yolo_open_world_exposure_per_seed.csv", index=False)
        if len(ydf):
            ys = ydf[ydf.species == "ALL"].groupby("group").rate.agg(["mean", "std", "count"]).reset_index()
            ys.to_csv(D_TAB / "T05b_yolo_open_world_exposure_mean_sd.csv", index=False)

    # ---------------- external / near-OOD / cross-source tests (R1-3, R1-4, R2-4) ----------------
    with guard('external / near-OOD / cross-source tests (R1-3, R1-4, R2-4)'):
        ext_rows, ext_auc = [], []
        if has_ext:
            for key in [k for k in scores if k[1] in ("full", "yoloroi", "image")]:
                s = scores[key]["score"]
                for pn in ["P1_val_known_TPR95", "P2_val_youden"]:
                    tau = taus[(key, pn)]
                    for grp in ["known_ext", "near_ood", "far_ext"]:
                        idx = I[grp]
                        for sp in ["ALL"] + sorted(set(sp_te[idx])):
                            ii = idx if sp == "ALL" else idx[sp_te[idx] == sp]
                            if len(ii) == 0:
                                continue
                            rej = int((s[ii] > tau).sum())
                            lo, hi = wilson(rej, len(ii))
                            ext_rows.append(dict(backbone=key[0], view=key[1], family=key[2], protocol=pn, group=grp,
                                                 species=sp, n=len(ii), rejected_as_unknown=rej,
                                                 rejection_rate=rej / len(ii), ci_lo=lo, ci_hi=hi,
                                                 desired=("ACCEPT (boar from another source)" if grp == "known_ext" else "REJECT")))
                pairs = [("known_test", "near_ood"), ("known_test", "far_ext"), ("known_ext", "near_ood"),
                         ("known_ext", "far_ext"), ("known_test", "known_ext"), ("known_test", "unknown_test")]
                nm = {"known_test": I["te_k"], "unknown_test": I["te_u"], **{k: I[k] for k in ["known_ext", "near_ood", "far_ext"]}}
                for a, bq in pairs:
                    if len(nm[a]) and len(nm[bq]):
                        ext_auc.append(dict(backbone=key[0], view=key[1], family=key[2], negatives=a, positives=bq,
                                            auroc=auroc(s[nm[a]], s[nm[bq]])))
        pd.DataFrame(ext_rows).to_csv(D_TAB / "T06_external_near_ood_cross_source.csv", index=False)
        pd.DataFrame(ext_auc).to_csv(D_TAB / "T06b_external_auroc_pairs.csv", index=False)

    # ---------------- background-bias analysis (R1-4) ----------------
    with guard('background-bias analysis (R1-4)'):
        bg_rows = []
        cover = man.box_cover.values
        for key in [k for k in scores if k[2] in ("evt_proto", "knn", "mahalanobis", "mp_evt")]:
            s = scores[key]["score"]
            for subset, mk, mu in [("all", I["te_k"], I["te_u"]),
                                   ("box_cover<0.5", I["te_k"][cover[I["te_k"]] < 0.5], I["te_u"][cover[I["te_u"]] < 0.5])]:
                if len(mk) < 5 or len(mu) < 5:
                    continue
                bsub = Boot(len(mk), len(mu), BOOTSTRAP_B, seed=SEED + 7)
                mt, _ = evaluate(s[mk], s[mu], taus[(key, "P1_val_known_TPR95")], bsub)
                bg_rows.append(dict(backbone=key[0], view=key[1], family=key[2], subset=subset,
                                    n_known=len(mk), n_unknown=len(mu), auroc=mt["auroc"], auroc_lo=mt["auroc_lo"],
                                    auroc_hi=mt["auroc_hi"], macro_f1=mt["macro_f1"], unknown_recall=mt["unknown_recall"],
                                    known_recall=mt["known_recall"]))
        # linear probe: can dataset SOURCE be predicted from backgrounds alone?
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import StratifiedKFold, cross_val_predict
        probe = []
        ids = np.r_[I["te_k"], I["te_u"], I["cal_u"]]
        yy = np.r_[np.zeros(len(I["te_k"])), np.ones(len(I["te_u"]) + len(I["cal_u"]))]
        for (b, v), X in avail.items():
            if v not in ("full", "bg", "gtroi"):
                continue
            for subset in ["all", "box_cover<0.5"]:
                mk = np.ones(len(ids), bool) if subset == "all" else cover[ids] < 0.5
                if mk.sum() < 20 or len(set(yy[mk])) < 2 or min(np.bincount(yy[mk].astype(int))) < 5:
                    continue
                Zp = l2n(X[ids[mk]].astype(np.float64))
                cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
                pr = cross_val_predict(LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced"),
                                       Zp, yy[mk], cv=cv, method="predict_proba")[:, 1]
                probe.append(dict(backbone=b, view=v, subset=subset, n=int(mk.sum()),
                                  source_probe_auroc_5fold=auroc(pr[yy[mk] == 0], pr[yy[mk] == 1])))
        pd.DataFrame(bg_rows).to_csv(D_TAB / "T07_background_bias_views.csv", index=False)
        pd.DataFrame(probe).to_csv(D_TAB / "T07b_background_source_linear_probe.csv", index=False)
        cov = man.groupby(["source", "split"]).box_cover.describe().reset_index()
        cov.to_csv(D_TAB / "T07c_box_coverage_by_split.csv", index=False)

    # ---------------- near-duplicate audit (leakage check) ----------------
    with guard('near-duplicate audit (leakage check)'):
        dup_rows, dedup_rows = [], []
        if ("vit_b16", "full") in avail:
            Z = l2n(avail[("vit_b16", "full")].astype(np.float64))

            def maxsim(q, bank):
                out = np.zeros(len(q))
                for i in range(0, len(q), 2048):
                    out[i:i + 2048] = (Z[q[i:i + 2048]] @ Z[bank].T).max(1)
                return out
            ms_te_k = maxsim(I["te_k"], I["train"])
            ms_te_u = maxsim(I["te_u"], I["cal_u"])
            ms_cal_k = maxsim(I["cal_k"], I["train"])
            for name, ms in [("boar test vs boar train", ms_te_k), ("boar val vs boar train", ms_cal_k),
                             ("wildlife test vs wildlife val", ms_te_u)]:
                dup_rows.append(dict(pair=name, n=len(ms), median_max_cos=float(np.median(ms)),
                                     n_ge_0_95=int((ms >= 0.95).sum()), n_ge_0_98=int((ms >= 0.98).sum()),
                                     n_ge_0_99=int((ms >= 0.99).sum())))
            keep_k = I["te_k"][ms_te_k < DUP_SIM]
            keep_u = I["te_u"][ms_te_u < DUP_SIM]
            if len(keep_k) > 5 and len(keep_u) > 5:
                bd = Boot(len(keep_k), len(keep_u), BOOTSTRAP_B, seed=SEED + 11)
                for key in [k for k in scores if k[0] in ("vit_b16", "yolov8n")]:
                    s = scores[key]["score"]
                    mt, _ = evaluate(s[keep_k], s[keep_u], taus[(key, "P1_val_known_TPR95")], bd)
                    keep = {k: v for k, v in mt.items() if k not in ("n_known", "n_unknown")
                            and (not k.endswith(("_lo", "_hi")) or k.startswith(("auroc", "macro_f1", "unknown_recall", "known_recall")))}
                    dedup_rows.append(dict(backbone=key[0], view=key[1], family=key[2],
                                           n_known=len(keep_k), n_unknown=len(keep_u), **keep))
        pd.DataFrame(dup_rows).to_csv(D_TAB / "T08_near_duplicate_audit.csv", index=False)
        pd.DataFrame(dedup_rows).to_csv(D_TAB / "T08b_results_on_deduplicated_test.csv", index=False)

    # ---------------- calibration variability (R1-5) ----------------
    with guard('calibration variability (R1-5)'):
        cal_rows = []
        if ("vit_b16", "full") in avail:
            X = avail[("vit_b16", "full")].astype(np.float64)
            Zf = l2n(X)
            rng = np.random.default_rng(SEED + 3)
            k_sel = scores.get(("vit_b16", "full", "knn"), {}).get("extra", {}).get("k", 10)
            K_sel = scores.get(("vit_b16", "full", "mp_evt"), {}).get("extra", {}).get("K", 4)
            for r in range(CALIB_REPEATS):
                sub = rng.choice(I["train"], int(0.8 * len(I["train"])), replace=False)
                mu = X[sub].mean(0)
                d = 1 - Zf @ (mu / np.linalg.norm(mu))
                fits = [fit_weibull_tail(d[sub], t) for t in TAILS]
                pks = min(fits, key=lambda p: p["ks"])
                pp = [p for p in fits if abs(p["tail_frac"] - PAPER_TAIL) < 1e-9][0]
                cand = {"evt_proto(KS tail)": wcdf(d, pks), "evt_proto(tail=0.20)": wcdf(d, pp),
                        f"knn(k={k_sel})": knn_scores(Zf, sub, [k_sel])[k_sel],
                        f"mp_evt(K={K_sel})": mp_evt_scores(Zf, sub, K_sel, pks["tail_frac"], seed=SEED + r, n_init=3)[0]}
                for name, s in cand.items():
                    for pn, tau in [("P1_val_known_TPR95", tau_p1(s[I["cal_k"]])),
                                    ("P2_val_youden", tau_youden(s[I["cal_k"]], s[I["cal_u"]]))] + \
                                   ([("P0_fixed_tau0.5", 0.5)] if name.startswith("evt") else []):
                        mt, _ = evaluate(s[I["te_k"]], s[I["te_u"]], tau, None)
                        cal_rows.append(dict(repeat=r, method=name, protocol=pn, tau=tau,
                                             tail_selected=pks["tail_frac"], weibull_k=pks["shape_k"],
                                             weibull_lambda=pks["scale_lambda"],
                                             **{k: mt[k] for k in ["auroc", "accuracy", "balanced_accuracy", "macro_f1",
                                                                   "mcc", "known_recall", "unknown_recall", "unknown_far"]}))
        cdf_ = pd.DataFrame(cal_rows)
        cdf_.to_csv(D_TAB / "T09_calibration_variability_repeats.csv", index=False)
        if len(cdf_):
            agg = cdf_.groupby(["method", "protocol"]).agg(
                ["mean", "std", "min", "max"])[["auroc", "macro_f1", "mcc", "known_recall", "unknown_recall",
                                                 "unknown_far", "tau", "weibull_k", "weibull_lambda", "tail_selected"]]
            agg.columns = ["_".join(c) for c in agg.columns]
            agg.reset_index().to_csv(D_TAB / "T09b_calibration_variability_summary.csv", index=False)

    # ---------------- threshold sensitivity (replaces old Table 9; descriptive only) ----------------
    with guard('threshold sensitivity (replaces old Table 9; descriptive only)'):
        ts_rows = []
        for key in [("vit_b16", "full", "evt_proto_paper"), ("vit_b16", "full", "evt_proto")]:
            if key not in scores:
                continue
            s = scores[key]["score"]
            for tau in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
                for split, kk, uu in [("val", I["cal_k"], I["cal_u"]), ("test", I["te_k"], I["te_u"])]:
                    mt, _ = evaluate(s[kk], s[uu], tau, None)
                    ts_rows.append(dict(method=key[2], config=scores[key]["config"], split=split, tau=tau,
                                        **{k: mt[k] for k in ["accuracy", "balanced_accuracy", "macro_f1", "mcc",
                                                              "known_recall", "unknown_recall", "unknown_far"]}))
        pd.DataFrame(ts_rows).to_csv(D_TAB / "T11_threshold_sensitivity_val_and_test.csv", index=False)

    # ---------------- YOLO detection summary and real training curves ----------------
    with guard('YOLO detection summary and real training curves'):
        yd = D_TAB / "T01_yolo_detection_per_seed.csv"
        if yd.exists():
            y = pd.read_csv(yd)
            y.groupby(["split", "conf"])[["precision", "recall", "mAP50", "mAP50_95"]].agg(["mean", "std"]) \
                .reset_index().to_csv(D_TAB / "T01b_yolo_detection_mean_sd.csv")
        curves = []
        for s in YOLO_SEEDS:
            rc = D_YOLO / f"seed{s}" / "results.csv"
            if rc.exists():
                c = pd.read_csv(rc)
                c.columns = [x.strip() for x in c.columns]
                c["seed"] = s
                curves.append(c)
        if curves:
            cv_ = pd.concat(curves)
            cv_.to_csv(D_TAB / "T12_yolo_training_curves_all_seeds.csv", index=False)
            ep = cv_.groupby("seed").epoch.max().reset_index(name="epochs_completed")
            ep.to_csv(D_TAB / "T12b_yolo_epochs_completed.csv", index=False)

    # ---------------- Weibull parameters (all backbones/views) ----------------
    with guard('Weibull parameters (all backbones/views)'):
        wrows = [dict(backbone=r["backbone"], view=r["view"], config=r["config"], ks=r["weibull_ks"],
                      shape_k=r["weibull_k"], scale_lambda=r["weibull_lambda"])
                 for r in grid_rows if r["family"] == "evt_proto" and r["protocol"] == "P1_val_known_TPR95"]
        pd.DataFrame(wrows).to_csv(D_TAB / "T13_weibull_parameters_and_KS.csv", index=False)

    # ---------------- per-sample scores (R1-7) ----------------
    with guard('per-sample scores (R1-7)'):
        ps = man[["img_id", "path", "source", "split", "group", "species", "n_boxes", "box_cover"]].copy()
        for key, d in scores.items():
            col = "__".join(key)
            ps["score__" + col] = d["score"]
            t1 = taus[(key, "P1_val_known_TPR95")]
            ps["pred_P1__" + col] = np.where(np.isnan(d["score"]), "", np.where(d["score"] > t1, "unknown", "known"))
        for sd, p in yolo.items():
            ps[f"yolo_seed{sd}_max_conf"] = p.max_conf.values
            ps[f"yolo_seed{sd}_n_det_025"] = p.n_det_025.values
        ps.to_csv(OUT / "per_sample_scores_and_predictions.csv", index=False)
        pd.DataFrame([dict(method="__".join(k[0]), protocol=k[1], tau=v) for k, v in taus.items()]) \
            .to_csv(D_TAB / "T14_frozen_thresholds_from_validation.csv", index=False)

    # ---------------- figures ----------------
    with guard("figures"):
        make_figures(man, I, scores, taus, res, curves)
    with guard("summary"):
        write_summary(res, I, has_ext)

In [ ]:
# %% ===================== 10. FIGURES (IJIES style: one plot per file, >=10pt, no (a)/(b)) ==
def make_figures(man, I, scores, taus, res, curves):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 11, "axes.labelsize": 11, "xtick.labelsize": 10,
                         "ytick.labelsize": 10, "legend.fontsize": 10, "figure.dpi": 100})

    def save(name):
        plt.tight_layout()
        plt.savefig(D_FIG / name, dpi=300, bbox_inches="tight")
        plt.close()

    # YOLO real training curves
    if curves:
        cv_ = pd.concat(curves)
        for cols, ylab, name in [(["train/box_loss", "train/cls_loss", "train/dfl_loss"], "Training loss", "F01_yolo_training_loss.png"),
                                 (["val/box_loss", "val/cls_loss", "val/dfl_loss"], "Validation loss", "F02_yolo_validation_loss.png"),
                                 (["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"], "Score", "F03_yolo_validation_metrics.png")]:
            plt.figure(figsize=(6.5, 4.2))
            for c in cols:
                if c not in cv_.columns:
                    continue
                st = cv_.groupby("epoch")[c].agg(["mean", "std", "count"]).reset_index()
                plt.plot(st.epoch, st["mean"], label=c.split("/")[-1].replace("(B)", ""))
                plt.fill_between(st.epoch, st["mean"] - st["std"].fillna(0), st["mean"] + st["std"].fillna(0), alpha=0.2)
            plt.xlabel("Epoch")
            plt.ylabel(ylab)
            plt.legend()
            plt.grid(alpha=0.3)
            save(name)

    # ROC curves on TEST
    from sklearn.metrics import roc_curve
    plt.figure(figsize=(6, 5))
    for key in [("vit_b16", "full", "evt_proto"), ("vit_b16", "full", "mahalanobis"), ("vit_b16", "full", "knn"),
                ("vit_b16", "full", "mp_evt"), ("vit_b16", "full", "ocsvm"), ("vit_b16", "gtroi", "evt_proto"),
                ("bioclip", "full", "knn"), ("dinov2_b14", "full", "knn"), ("yolov8n", "image", "yolo_conf")]:
        if key not in scores:
            continue
        s = scores[key]["score"]
        y = np.r_[np.zeros(len(I["te_k"])), np.ones(len(I["te_u"]))]
        fpr, tpr, _ = roc_curve(y, np.r_[s[I["te_k"]], s[I["te_u"]]])
        plt.plot(fpr, tpr, label=f"{key[0]}/{key[1]}/{key[2]} ({auroc(s[I['te_k']], s[I['te_u']]):.3f})")
    plt.plot([0, 1], [0, 1], "k--", lw=0.8)
    plt.xlabel("False positive rate (known rejected)")
    plt.ylabel("True positive rate (unknown rejected)")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.3)
    save("F04_roc_test_methods.png")

    # score distributions with frozen P1 threshold
    key = ("vit_b16", "full", "evt_proto")
    if key in scores:
        s = scores[key]["score"]
        plt.figure(figsize=(6.5, 4.2))
        plt.hist(s[I["te_k"]], bins=40, alpha=0.6, density=True, label="Known wild boar (test)")
        plt.hist(s[I["te_u"]], bins=40, alpha=0.6, density=True, label="Unknown wildlife (test)")
        plt.axvline(taus[(key, "P1_val_known_TPR95")], color="k", ls="--", label="Threshold frozen on validation (P1)")
        plt.xlabel("EVT unknown score")
        plt.ylabel("Density")
        plt.legend()
        save("F05_evt_score_distribution_test.png")
        # Weibull fit plot
        ex = scores[key]["extra"]
        X = np.load(D_FEAT / "vit_b16__full.npy").astype(np.float64)
        d = proto_dist(X, I["train"])
        tail = np.sort(d[I["train"]])[-ex["tail_n"]:]
        xs = np.linspace(tail.min() * 0.9, max(1.0, d.max()), 300)
        plt.figure(figsize=(6.5, 4.2))
        plt.hist(tail, bins=40, density=True, alpha=0.5, label="Extreme-tail distances (train)")
        plt.plot(xs, weibull_min.pdf(xs, ex["shape_k"], 0, ex["scale_lambda"]), label="Weibull PDF")
        ax2 = plt.gca().twinx()
        ax2.plot(xs, weibull_min.cdf(xs, ex["shape_k"], 0, ex["scale_lambda"]), "r--", label="Weibull CDF")
        ax2.set_ylabel("CDF")
        plt.gca().set_xlabel("Cosine distance to prototype")
        save("F06_weibull_fit.png")

    # confusion matrices at P1 and P2 (separate files)
    if key in scores:
        s = scores[key]["score"]
        for pn in ["P1_val_known_TPR95", "P2_val_youden"]:
            tau = taus[(key, pn)]
            cm = np.array([[int((s[I["te_k"]] <= tau).sum()), int((s[I["te_k"]] > tau).sum())],
                           [int((s[I["te_u"]] <= tau).sum()), int((s[I["te_u"]] > tau).sum())]])
            plt.figure(figsize=(4.6, 4))
            plt.imshow(cm, cmap="Blues")
            for i in range(2):
                for j in range(2):
                    plt.text(j, i, f"{cm[i, j]}\n({100 * cm[i, j] / cm[i].sum():.1f}%)", ha="center", va="center",
                             color="white" if cm[i, j] > cm.max() / 2 else "black")
            plt.xticks([0, 1], ["Known", "Unknown"])
            plt.yticks([0, 1], ["Known", "Unknown"])
            plt.xlabel("Predicted")
            plt.ylabel("True")
            save(f"F07_confusion_{pn}.png")

    # species-wise recall with CI
    sp = pd.read_csv(D_TAB / "T04_species_wise_unknown_recall.csv")
    sub = sp[(sp.backbone == "vit_b16") & (sp.view == "full") & (sp.family == "evt_proto") &
             (sp.protocol == "P1_val_known_TPR95") & (sp.split == "test")]
    if len(sub):
        plt.figure(figsize=(6.5, 4))
        x = np.arange(len(sub))
        plt.bar(x, sub.unknown_recall * 100, yerr=[(sub.unknown_recall - sub.ci_lo) * 100, (sub.ci_hi - sub.unknown_recall) * 100], capsize=4)
        plt.xticks(x, [f"{a}\n(n={n})" for a, n in zip(sub.species, sub.n)])
        plt.ylabel("Unknown recall (%)")
        plt.ylim(0, 105)
        save("F08_species_unknown_recall.png")

    # external groups
    ef = D_TAB / "T06_external_near_ood_cross_source.csv"
    if ef.exists() and ef.stat().st_size > 10:
        e = pd.read_csv(ef)
        e = e[(e.protocol == "P1_val_known_TPR95") & (e.species != "ALL")]
        for key in [("vit_b16", "full", "evt_proto"), ("vit_b16", "yoloroi", "pipeline_yolo_evt"), ("bioclip", "full", "knn")]:
            q = e[(e.backbone == key[0]) & (e.view == key[1]) & (e.family == key[2])]
            if not len(q):
                continue
            q = q.sort_values(["group", "species"])
            plt.figure(figsize=(9, 4.2))
            x = np.arange(len(q))
            cols = ["tab:green" if gname == "known_ext" else ("tab:orange" if gname == "near_ood" else "tab:red") for gname in q.group]
            plt.bar(x, q.rejection_rate * 100, color=cols,
                    yerr=[(q.rejection_rate - q.ci_lo) * 100, (q.ci_hi - q.rejection_rate) * 100], capsize=2)
            plt.xticks(x, q.species, rotation=60, ha="right")
            from matplotlib.patches import Patch
            plt.legend(handles=[Patch(color="tab:green", label="Boar, other source (should be accepted)"),
                                Patch(color="tab:orange", label="Near-OOD (should be rejected)"),
                                Patch(color="tab:red", label="Far-OOD, other source (should be rejected)")],
                       loc="upper right")
            plt.ylabel("Rejected as unknown (%)")
            plt.ylim(0, 105)
            save(f"F09_external_rejection_{'_'.join(key)}.png")

    # background-bias bars
    bf = pd.read_csv(D_TAB / "T07_background_bias_views.csv")
    q = bf[(bf.family == "evt_proto") & (bf.subset == "all")]
    if len(q):
        plt.figure(figsize=(7.5, 4))
        lab = [f"{a}\n{b}" for a, b in zip(q.backbone, q.view)]
        x = np.arange(len(q))
        plt.bar(x, q.auroc, yerr=[q.auroc - q.auroc_lo, q.auroc_hi - q.auroc], capsize=3)
        plt.xticks(x, lab, rotation=45, ha="right")
        plt.ylabel("Test AUROC (EVT score)")
        plt.ylim(0.4, 1.01)
        plt.axhline(0.5, color="k", lw=0.8, ls="--")
        save("F10_background_bias_auroc_by_view.png")

    # threshold sensitivity (validation and test)
    tf = pd.read_csv(D_TAB / "T11_threshold_sensitivity_val_and_test.csv")
    for split in ["val", "test"]:
        q = tf[(tf.method == "evt_proto_paper") & (tf.split == split)]
        if not len(q):
            continue
        plt.figure(figsize=(6.5, 4.2))
        for c, lb in [("accuracy", "Accuracy"), ("known_recall", "Known recall"), ("unknown_recall", "Unknown recall"), ("macro_f1", "Macro-F1")]:
            plt.plot(q.tau, q[c] * 100, marker="o", label=lb)
        plt.xlabel("EVT threshold")
        plt.ylabel("%")
        plt.legend()
        plt.grid(alpha=0.3)
        save(f"F11_threshold_sensitivity_{split}.png")


def write_summary(res, I, has_ext):
    L = ["# OpenWild-BoarNet revision — key results (auto-generated)", "",
         f"Smoke test: {SMOKE_TEST}", f"Images: " + ", ".join(f"{k}={len(v)}" for k, v in I.items()), "",
         "All thresholds and hyper-parameters were fixed on validation data (boar val + wildlife valid); "
         "the test split was used once. Values: point [95% bootstrap CI].", ""]
    f2 = D_TAB / "T02b_all_methods_TEST_formatted.csv"
    if f2.exists():
        t = pd.read_csv(f2)
        t = t[t.protocol == "P1_val_known_TPR95"]
        L.append("## Main table (P1: threshold = 95% known acceptance on validation)")
        try:
            L.append(t.to_markdown(index=False))
        except Exception:
            L.append(t.to_string(index=False))
    for f in ["T01b_yolo_detection_mean_sd.csv", "T05b_yolo_open_world_exposure_mean_sd.csv", "T08_near_duplicate_audit.csv",
              "T07b_background_source_linear_probe.csv", "T09b_calibration_variability_summary.csv", "T12b_yolo_epochs_completed.csv"]:
        p = D_TAB / f
        if p.exists() and p.stat().st_size > 5:
            L += ["", f"## {f}", pd.read_csv(p).to_string(index=False)]
    (OUT / "RESULTS_SUMMARY.md").write_text("\n".join(L))
    log("summary written")

In [ ]:
# %% ===================== 11. PACKAGE OUTPUTS =================================
def stage_package():
    import platform
    env = {"python": platform.python_version(), "SMOKE_TEST": SMOKE_TEST}
    for p in ["torch", "torchvision", "ultralytics", "timm", "open_clip_torch", "numpy", "scipy",
              "scikit-learn", "pandas"]:
        env[p] = _ver(p)
    try:
        import torch
        env["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    except Exception:
        pass
    cfg = {k: globals()[k] for k in ["SEED", "YOLO_SEEDS", "YOLO_EPOCHS", "YOLO_PATIENCE", "YOLO_IMGSZ", "YOLO_BATCH",
                                     "BOOTSTRAP_B", "CALIB_REPEATS", "TAILS", "PAPER_TAIL", "KNN_K", "MP_K", "OCSVM_NU",
                                     "TPR_TARGET", "YOLO_CONF", "ROI_PAD", "DUP_SIM", "NEAR_OOD", "FAR_EXT", "KNOWN_EXT",
                                     "BACKBONES", "VIEWS"]}
    json.dump({"environment": env, "config": cfg}, open(OUT / "config_and_environment.json", "w"), indent=2, default=str)
    wdir = OUT / "weights"
    wdir.mkdir(exist_ok=True)
    for s in YOLO_SEEDS:
        b = D_YOLO / f"seed{s}" / "weights" / "best.pt"
        if b.exists():
            shutil.copy(b, wdir / f"yolov8n_wildboar_seed{s}_best.pt")
    zp = WORK / ("revision_results_smoke.zip" if SMOKE_TEST else "revision_results.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob("*"):
            if p.is_file() and "features" not in p.parts and p.suffix not in (".pt",) and \
                    not (p.suffix == ".jpg" and "eval" in p.parts and "train_batch" in p.name):
                z.write(p, p.relative_to(WORK))
    log(f"results zip: {zp}  ({zp.stat().st_size / 1e6:.1f} MB). Weights are in {wdir}")

In [ ]:
# %% ===================== 12. RUN EVERYTHING ===================================
if __name__ == "__main__" or IN_KAGGLE:
    if not os.environ.get("OWB_NO_AUTORUN"):
        T0 = time.time()
        log(f"SMOKE_TEST={SMOKE_TEST}  CUDA={cuda_ok()}  OUT={OUT}")
        run_stage("check", stage_check)
        if not json.loads(STATUS_FILE.read_text()).get("check", {}).get("ok", False):
            raise SystemExit("Environment check failed - see the message above. Nothing else was run.")
        run_stage("discover", stage_discover)
        run_stage("yolo_train", stage_yolo_train)
        run_stage("yolo_eval", stage_yolo_eval)
        run_stage("features", stage_features)
        run_stage("analysis", stage_analysis)
        run_stage("package", stage_package)
        log(f"TOTAL {(time.time() - T0) / 60:.1f} min")
        print(open(STATUS_FILE).read())